# Task 1 — Motivation: Why Sequences Matter

### The Sequential Nature of Natural Language
Natural language is inherently sequential. Words do not exist in isolation; their meaning is deeply tied to their position in a sentence and their relationship to adjacent words. In any human language, the order of words is just as important as the words themselves.

### Why Bag-of-Words and TF-IDF Fail on Sequences
Classical text representation models like **Bag-of-Words (BoW)** and **TF-IDF** represent text by building simple word frequency tables. They treat a document as an unordered collection (a "bag") of tokens. While this statistical count works for simple document classification (e.g. spam detection where words like "free" or "win" are strong indicators), it fails completely on sequence-dependent tasks due to several major blind spots:

1. **Loss of Spatial Structure**: Discarding word order means losing the context window connections between adjacent words.
2. **Permutation Traps (Identical Vectors)**: Sentences that contain the exact same vocabulary but mean opposite things (e.g., `"dog bites man"` vs. `"man bites dog"`) map to the exact same frequency indices in a sparse vector. Their cosine similarity is calculated as exactly `1.0`, meaning a classical model sees them as identical statement representations.
3. **Negation Scope Loss**: Negation words like `"not"` or `"no"` modify the words around them. In a bag-of-words representation, `"not"` is treated as a separate feature, completely disconnected from the adjective it qualifies. Thus, `"not good, actually bad"` is treated the same as `"not bad, actually good"`.

![Sequence Blindspot](outputs/bow_sequence_blindspot.png)

### Transitioning to Temporal Sequences
To address these blind spots, we must use models that process words *in order*, maintaining an internal memory state that updates with each incoming token. This is where Recurrent Neural Networks (RNNs) come into the picture.

### Core Use Cases
- **Sentiment Analysis**: Detecting subtle shifts in sentiment caused by word order (e.g. `"not good, actually bad"` vs `"not bad, actually good"`).
- **Instruction Parsing**: Understanding command hierarchies where the target changes based on word order (e.g. `"send email to manager from assistant"` vs `"send email to assistant from manager"`).


### Step 1 — Vectorize Word-Reversed Sentences


In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Define 5 pairs of sentences with identical vocabularies but different meanings
sentence_pairs = [
    ("dog bites man", "man bites dog"),
    ("not good, actually bad", "not bad, actually good"),
    ("the movie was bad, not good", "the movie was good, not bad"),
    ("assistant sent email to manager", "manager sent email to assistant"),
    ("he failed because he didn't work", "he worked because he didn't fail")
]

all_sentences = []
for p in sentence_pairs:
    all_sentences.extend(p)

# Fit CountVectorizer (BoW)
bow_vec = CountVectorizer()
bow_matrix = bow_vec.fit_transform(all_sentences)

# Fit TfidfVectorizer
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(all_sentences)

# Evaluate cosine similarities within each pair
bow_similarities = []
tfidf_similarities = []

for i in range(len(sentence_pairs)):
    idx1 = 2 * i
    idx2 = 2 * i + 1
    
    # Calculate similarities
    bow_sim = cosine_similarity(bow_matrix[idx1], bow_matrix[idx2])[0][0]
    tfidf_sim = cosine_similarity(tfidf_matrix[idx1], tfidf_matrix[idx2])[0][0]
    
    bow_similarities.append(bow_sim)
    tfidf_similarities.append(tfidf_sim)

df_results = pd.DataFrame({
    'Sentence 1': [p[0] for p in sentence_pairs],
    'Sentence 2': [p[1] for p in sentence_pairs],
    'BoW Cosine Sim': bow_similarities,
    'TF-IDF Cosine Sim': tfidf_similarities
})

display(df_results)



,Sentence 1,Sentence 2,BoW Cosine Sim,TF-IDF Cosine Sim
0,dog bites man,man bites dog,1.00,1.00000
1,"not good, actually bad","not bad, actually good",1.00,1.00000
2,"the movie was bad, not good","the movie was good, not bad",1.00,1.00000
3,assistant sent email to manager,manager sent email to assistant,1.00,1.00000
4,he failed because he didn't work,he worked because he didn't fail,0.75,0.68434


### Step 2 — User-Defined Interactive Evaluation Function
Enter two custom sentences below to inspect their Bag-of-Words cosine similarity and witness the limitation first-hand.


In [2]:
def evaluate_bow_limitation(sent1, sent2):
    """
    User-defined evaluation function. Compares BoW representation and Cosine Similarity.
    """
    vectorizer = CountVectorizer()
    vectors = vectorizer.fit_transform([sent1, sent2]).toarray()
    vocab = vectorizer.get_feature_names_out()
    
    df_vecs = pd.DataFrame(vectors, columns=vocab, index=['Sentence 1', 'Sentence 2'])
    sim = cosine_similarity(vectors[0].reshape(1, -1), vectors[1].reshape(1, -1))[0][0]
    
    print("--- Vocabulary Vectors ---")
    display(df_vecs)
    print(f"\nCosine Similarity: {sim:.4f}")
    
    if sim > 0.95:
        print("\nObservation: The similarity is near 1.0! This proves the model is blind to word order.")
        print("Although the sentences have different meanings, the Bag-of-Words representation treats them as identical.")
    else:
        print("\nObservation: The vocabularies differ, so the vectors are distinct.")

# Evaluate with custom sentences
evaluate_bow_limitation("I love natural language processing", "natural language processing I love")



--- Vocabulary Vectors ---


,language,love,natural,processing
Sentence 1,1,1,1,1
Sentence 2,1,1,1,1



Cosine Similarity: 1.0000

Observation: The similarity is near 1.0! This proves the model is blind to word order.
Although the sentences have different meanings, the Bag-of-Words representation treats them as identical.
